# BMW Repair Order | GEPA Prompt Optimization POC

What this does in order:
1. Upload PDF + JSON pairs from `Data/Samples/`
2. Run a basic prompt on the PDFs → score the JSON extraction (baseline)
3. GEPA automatically rewrites the prompt using the eval feedback
4. Re-score with the new prompt → show the improvement

Same model, same documents, only the prompt changes. That delta is the result.

In [1]:
%pip install openai groq gepa litellm PyMuPDF deepdiff transformers accelerate pillow --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.4/191.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 142.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 32.1 MB/s eta 0:00:00


## Config

In [ ]:
import os

BACKEND = 'local'  # 'local' = runs on Colab GPU, no API key needed

GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY'   # get free key at aistudio.google.com
GROQ_API_KEY   = 'YOUR_GROQ_API_KEY'     # get free key at console.groq.com

GEPA_MAX_CALLS = 30  # extraction calls GEPA gets (each one = one model call)
DATA_DIR = '/content/samples'

In [3]:
# set up client based on chosen backend
from openai import OpenAI

if BACKEND == 'gemini':
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    os.environ['GOOGLE_API_KEY'] = GEMINI_API_KEY  # litellm uses this
    client        = OpenAI(api_key=GEMINI_API_KEY,
                           base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    EXTRACT_MODEL = 'gemini-2.0-flash'
    REFLECTION_LM = 'gemini/gemini-2.0-flash'
    _use_local    = False

elif BACKEND == 'groq':
    os.environ['GROQ_API_KEY'] = GROQ_API_KEY
    client        = OpenAI(api_key=GROQ_API_KEY, base_url='https://api.groq.com/openai/v1')
    EXTRACT_MODEL = 'meta-llama/llama-4-scout-17b-16e-instruct'
    REFLECTION_LM = 'groq/llama-3.3-70b-versatile'
    _use_local    = False

elif BACKEND == 'local':
    import litellm
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, pipeline as hf_pipeline
    import torch
    print('Loading Qwen2-VL-2B... (takes a few minutes)')
    _vl_model = Qwen2VLForConditionalGeneration.from_pretrained(
        'Qwen/Qwen2-VL-2B-Instruct', torch_dtype=torch.float16, device_map='auto'
    )
    _vl_proc      = AutoProcessor.from_pretrained('Qwen/Qwen2-VL-2B-Instruct')
    _reflect_pipe = hf_pipeline('text-generation', model='Qwen/Qwen2.5-3B-Instruct',
                                 torch_dtype=torch.float16, device_map='auto', max_new_tokens=1024)
    def _local_litellm(model, messages, **kwargs):
        prompt = '\n'.join(m['content'] for m in messages if isinstance(m['content'], str))
        out    = _reflect_pipe(prompt)[0]['generated_text']
        class _R: pass
        r, c, m   = _R(), _R(), _R()
        m.content = out[len(prompt):].strip()
        c.message = m
        r.choices = [c]
        return r
    litellm.completion = _local_litellm
    client        = None
    REFLECTION_LM = 'local/qwen'
    _use_local    = True
    print('Models loaded.')

else:
    raise ValueError('BACKEND must be gemini, groq, or local')

print('Backend:', BACKEND, '|', 'Model:', EXTRACT_MODEL if not _use_local else 'local Qwen2-VL')

Loading Qwen2-VL-2B... (takes a few minutes)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Models loaded.
Backend: local | Model: local Qwen2-VL


## 1. Upload your files

In [4]:
import json, glob, os
import fitz
import base64
from pathlib import Path

os.makedirs(DATA_DIR, exist_ok=True)

from google.colab import files
uploaded = files.upload() # upload all PDF + JSON pairs from Data/Samples/

for name, data in uploaded.items():
    with open(os.path.join(DATA_DIR, name), 'wb') as f:
        f.write(data)

print('Saved', len(uploaded), 'files')

Saving 201414.json to 201414.json
Saving 201414.pdf to 201414.pdf
Saving 344098.json to 344098.json
Saving 344098.pdf to 344098.pdf
Saving 629903.json to 629903.json
Saving 629903.pdf to 629903.pdf
Saving 678856.json to 678856.json
Saving 678856.pdf to 678856.pdf
Saving 809570.json to 809570.json
Saving 809570.pdf to 809570.pdf
Saving 944962.json to 944962.json
Saving 944962.pdf to 944962.pdf
Saving overview.pdf to overview.pdf
Saved 13 files


## 2. Load docs + render pages
PDF pages are rendered once as PNG images. GEPA will call extraction many times so we don't want to re-render on every call.

In [5]:
def load_samples(data_dir):
    samples = []
    for jpath in sorted(glob.glob(data_dir + '/*.json')):
        doc_id = Path(jpath).stem
        pdf_path = data_dir + '/' + doc_id + '.pdf'
        if not os.path.exists(pdf_path): continue
        with open(jpath) as f:
            gt = json.load(f)
        samples.append({'doc_id': doc_id, 'pdf_path': pdf_path, 'ground_truth': gt})
    return samples

def pdf_to_images(pdf_path, dpi=150):
    doc, imgs = fitz.open(pdf_path), []
    for page in doc:
        pix = page.get_pixmap(matrix=fitz.Matrix(dpi/72, dpi/72))
        imgs.append(base64.b64encode(pix.tobytes('png')).decode())
    doc.close()
    return imgs

samples    = load_samples(DATA_DIR)
doc_images = {s['doc_id']: pdf_to_images(s['pdf_path']) for s in samples}

print('Loaded', len(samples), 'docs')
for s in samples:
    print('  ' + s['doc_id'] + ': ' + str(len(doc_images[s['doc_id']])) + ' pages')

Loaded 6 docs
  201414: 6 pages
  344098: 4 pages
  629903: 5 pages
  678856: 9 pages
  809570: 6 pages
  944962: 10 pages


## 3. Eval + extraction
Just run these two cells — eval is inlined from `Scripts/eval.py`, extraction sends images to the model.

In [6]:
from deepdiff import DeepDiff

WEIGHTS = {'structure': 0.45, 'numbers': 0.40, 'text': 0.15}
PENALTIES = {
    'structure': {'missing': 0.08, 'extra': 0.04, 'type': 0.06, 'value': 0.03},
    'numbers': {'missing': 0.08, 'extra': 0.02, 'type': 0.06, 'value': 0.06},
    'text': {'missing': 0.05, 'extra': 0.02, 'type': 0.03, 'value': 0.04},
}
CAPS = {'structure': 0.90, 'numbers': 0.90, 'text': 0.80}

def _is_num(x): return isinstance(x, (int, float)) and not isinstance(x, bool)
def _humanize(p): return p[4:].replace("['", '.').replace("']", '').lstrip('.') if p.startswith('root') else p
def _cat(path, exp, got, kind):
    if kind in ('missing', 'extra'):
        if _is_num(exp) or _is_num(got): return 'numbers'
        return 'text' if isinstance(exp, str) or isinstance(got, str) else 'structure'
    if _is_num(exp) or _is_num(got): return 'numbers'
    return 'text' if isinstance(exp, str) or isinstance(got, str) else 'structure'
def _norm(o):
    if isinstance(o, dict): return {k: _norm(v) for k, v in o.items()}
    if isinstance(o, list): return [_norm(v) for v in o]
    if isinstance(o, str):  y = ' '.join(o.split()); return None if y == '' else y
    return o

def evaluate_extraction(gt, pred):
    dd     = DeepDiff(_norm(gt), _norm(pred), view='tree',
                      ignore_numeric_type_changes=True, significant_digits=2, verbose_level=2)
    issues = []
    def emit(kind, path, exp, got):
        cat = _cat(path, exp, got, kind)
        issues.append({'category': cat, 'kind': kind, 'path': path,
                       'expected': exp, 'got': got, 'penalty': float(PENALTIES[cat].get(kind, 0.03))})
    for i in dd.get('values_changed',         []): emit('value',   _humanize(i.path()), i.t1, i.t2)
    for i in dd.get('type_changes',           []): emit('type',    _humanize(i.path()), i.t1, i.t2)
    for i in dd.get('dictionary_item_removed',[]):
        if i.t1 is not None: emit('missing', _humanize(i.path()), i.t1, None)
    for i in dd.get('dictionary_item_added',  []):
        if i.t2 is not None: emit('extra',   _humanize(i.path()), None, i.t2)
    for i in dd.get('iterable_item_removed',  []): emit('missing', _humanize(i.path()), i.t1, None)
    for i in dd.get('iterable_item_added',    []): emit('extra',   _humanize(i.path()), None, i.t2)
    raw   = {}
    for it in issues: raw[it['category']] = raw.get(it['category'], 0.0) + it['penalty']
    capped = {c: min(raw.get(c, 0.0), CAPS[c]) for c in CAPS}
    subs   = {c: max(0.0, 1.0 - capped.get(c, 0.0)) for c in WEIGHTS}
    return {'score':   round(sum(WEIGHTS[c] * subs[c] for c in WEIGHTS), 4),
            'subscores': subs,
            'issues':  sorted(issues, key=lambda x: -x['penalty'])}

def build_feedback(report, n=8):
    s     = report['subscores']
    lines = ['Score: ' + str(round(report['score'], 3)),
             '  structure=' + str(round(s.get('structure',1),2)) +
             '  numbers='  + str(round(s.get('numbers',1),2)) +
             '  text='     + str(round(s.get('text',1),2))]
    for iss in report.get('issues', [])[:n]:
        lines.append('  [' + iss['category'] + '][' + iss['kind'] + '] ' + iss['path'])
        lines.append('    expected: ' + repr(iss['expected'])[:60] + '  got: ' + repr(iss['got'])[:60])
    return '\n'.join(lines)

In [7]:
import torch
from PIL import Image
import io

BASE_PROMPT = (
    'You are extracting structured data from BMW repair order PDFs.\n\n'
    'Return a JSON object with doc_id (the RO number) and a sections array.\n'
    'Each section has: section_id, prefix (ASI/BWO/CSI/JSI/WSI/ISI), page_count, header, footer, content.\n\n'
    'Header fields: ro_number, vin, customer_name, customer_address, customer_city_state_zip,\n'
    'home_phone, contact_phone, advisor_id, advisor_name, color, year, make, model, submodel,\n'
    'engine, miles_in, miles_out, tag, open_date, close_date, deal_type, pay_type, rate,\n'
    'stock_number, unit_number, customer_number, time_in, date_in, time_out, date_out.\n\n'
    'Footer fields: labor_amount, parts_amount, gas_oil_lube, sublet_amount, misc_charges,\n'
    'total_charges, less_insurance, sales_tax, please_pay.\n\n'
    'Use null for missing fields. Numbers exact. Return ONLY the JSON object.'
)

# image limits per backend
MAX_IMGS = 5 if BACKEND == 'groq' else 16  # gemini handles up to 16 easily; groq caps at 5

def extract_from_pdf(images, system_prompt):
    if _use_local:
        # run through local Qwen2-VL
        pil_imgs = [Image.open(io.BytesIO(base64.b64decode(img))) for img in images]
        msgs = [{'role': 'user', 'content': [
            {'type': 'text', 'text': system_prompt + '\n\nExtract from this repair order:'},
        ] + [{'type': 'image', 'image': img} for img in pil_imgs]}]
        text = _vl_proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = _vl_proc(text=text, images=pil_imgs, return_tensors='pt').to(_vl_model.device)
        out = _vl_model.generate(**inputs, max_new_tokens=2048)
        raw = _vl_proc.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    else:
        content = [{'type': 'text', 'text': 'Extract all data from this BMW repair order:'}]
        for img in images[:MAX_IMGS]:
            content.append({'type': 'image_url',
                            'image_url': {'url': 'data:image/png;base64,' + img, 'detail': 'high'}})
        try:
            resp = client.chat.completions.create(
                model = EXTRACT_MODEL,
                messages=[{'role': 'system', 'content': system_prompt},
                           {'role': 'user',   'content': content}],
                max_tokens=4096, temperature=0.0,
            )
            raw = resp.choices[0].message.content.strip()
        except Exception as e:
            print('API error: ' + str(e))
            return {}
    try:
        start, end = raw.find('{'), raw.rfind('}')
        return json.loads(raw[start:end+1]) if start != -1 else {}
    except json.JSONDecodeError:
        return {}

## 4. Baseline

In [8]:
def run_eval(samples, prompt, label='run'):
    results = []
    for s in samples:
        print('[' + label + '] ' + s['doc_id'] + '...', end=' ', flush=True)
        pred   = extract_from_pdf(doc_images[s['doc_id']], prompt)
        if not pred: print('FAILED'); results.append({'doc_id': s['doc_id'], 'score': 0.0}); continue
        report = evaluate_extraction(s['ground_truth'], pred)
        print('score=' + str(report['score']))
        results.append({'doc_id': s['doc_id'], 'score': report['score'], 'report': report, 'pred': pred})
    avg = sum(r['score'] for r in results) / len(results) if results else 0
    print('\naverage: ' + str(round(avg, 4)))
    return results

baseline = run_eval(samples, BASE_PROMPT, label='baseline')

[baseline] 201414... score=0.838
[baseline] 344098... FAILED
[baseline] 629903... score=0.862
[baseline] 678856... FAILED
[baseline] 809570... FAILED
[baseline] 944962... FAILED

average: 0.2833


## 5. GEPA optimization
Just run both cells. GEPA will print its progress as it evolves the prompt.

In [9]:
import ast, random
import gepa as gepa_pkg

try:    from gepa.core.adapter import GEPAAdapter, EvaluationBatch
except: from gepa.adapter       import GEPAAdapter, EvaluationBatch

class ExtractionAdapter(GEPAAdapter):

    def _coerce(self, ex):
        if isinstance(ex, dict): return ex
        try:    return json.loads(ex)
        except: pass
        try:    return ast.literal_eval(ex)
        except: pass
        raise TypeError(str(type(ex)))

    def evaluate(self, batch, candidate, capture_traces=False):
        prompt = candidate.get('system_prompt', BASE_PROMPT)
        scores, outputs, traces = [], [], []
        for ex in batch:
            ex   = self._coerce(ex)
            pred = extract_from_pdf(doc_images[ex['doc_id']], prompt)
            if pred:
                rep      = evaluate_extraction(ex['ground_truth'], pred)
                score, fb = rep['score'], build_feedback(rep)
            else:
                score, fb = 0.0, 'No valid JSON returned.'
            scores.append(score)
            outputs.append(json.dumps(pred)[:400])
            if capture_traces:
                traces.append({'doc_id': ex['doc_id'], 'score': score, 'feedback': fb,
                               'prompt_preview': prompt[:200], 'output': json.dumps(pred)[:400]})
        return EvaluationBatch(scores=scores, outputs=outputs,
                               trajectories=traces if capture_traces else None)

    def make_reflective_dataset(self, candidate, eval_batch, components_to_update):
        records = []
        for tr in (eval_batch.trajectories or []):
            records.append({
                'Inputs':           {'task': 'Extract JSON from BMW repair order PDF',
                                     'doc': tr.get('doc_id',''), 'prompt': tr.get('prompt_preview','')},
                'Generated Outputs': tr.get('output', ''),
                'Feedback':          {'evaluation': tr.get('feedback', '')},
            })
        return {comp: records for comp in components_to_update}

In [10]:
random.seed(42)
random.shuffle(samples)
trainset = [{'doc_id': s['doc_id'], 'ground_truth': s['ground_truth']} for s in samples[:4]]
valset   = [{'doc_id': s['doc_id'], 'ground_truth': s['ground_truth']} for s in samples[4:]]

print('Training on:   ' + str([d['doc_id'] for d in trainset]))
print('Validation on: ' + str([d['doc_id'] for d in valset]))
print('\nRunning GEPA (' + str(GEPA_MAX_CALLS) + ' calls)...\n')

result = gepa_pkg.optimize(
    seed_candidate={'system_prompt': BASE_PROMPT},
    adapter=ExtractionAdapter(),
    trainset=trainset,
    valset=valset,
    reflection_lm=REFLECTION_LM,
    max_metric_calls=GEPA_MAX_CALLS,
)

best_prompt = result.best_candidate.get('system_prompt', BASE_PROMPT)
print('\nEvolved prompt:\n' + '='*60)
print(best_prompt)
print('='*60)

Training on:   ['678856', '344098', '629903', '809570']
Validation on: ['201414', '944962']

Running GEPA (30 calls)...

Iteration 0: Base program full valset score: 0.419 over 2 / 2 examples
Iteration 1: Selected program 0 score: 0.419


Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Iteration 1: Proposed new text for system_prompt: ```

```
```
You are extracting structured data from BMW repair order PDFs.

Return a JSON object with `doc_id` (the RO number) and a `sections` array.
Each section has: `section_id`, `prefix` (ASI/BWO/CSI/JSI/WSI/ISI), `page_count`, `header`, `footer`, and `content`.

The `header` field contains:
- `ro_number`: The repair order number.
- `vin`: Vehicle Identification Number.
- `customer_name`: The name of the customer.
- `customer_address`: The address of the customer.
- `customer_city_state_zip`: The city and state ZIP code of the customer.
- `home_phone`: The home phone number of the customer.
- `contact_phone`: The contact phone number of the customer.
- `advisor_id`: The ID of the service advisor.
- `advisor_name`: The name of the service advisor.
- `color`: The color of the vehicle.
- `year`: The production year of the vehicle.
- `make`: The manufacturer of the vehicle.
- `model`: The model of the vehicle.
- `submodel`: The submod

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Iteration 2: Proposed new text for system_prompt: You're tasked with extracting structured data from BMW repair order PDFs. Your job is to return a JSON object containing the document ID (RO number) and a sections array. Each section should include section_id, prefix (ASI/BWO/CSI/JSI/WSI/ISI), page_count, header, footer, and content. 

The header should contain fields such as ro_number, vin, customer_name, customer_address, customer_city_state_zip, home_phone, contact_phone, advisor_id, advisor_name, color, year, make, model, submodel, engine, miles_in, miles_out, tag, open_date, close_date, deal_type, pay_type, rate, stock_number, unit_number, customer_number, time_in, date_in, time_out, date_out. The footer should contain fields like labor_amount, parts_amount, gas_oil_lube, sublet_amount, misc_charges, total_charges, less_insurance, sales_tax, please_pay. If a field is missing, return null for its value. All numbers should be exact.

Please ensure your output is a single JSON object

...

Iteration 3: Proposed new text for system_prompt: # Example 1
## Inputs
### task
Extract JSON from BMW repair order PDF

### doc
678856

### prompt
You are extracting structured data from BMW repair order PDFs.

Return a JSON object with doc_id (the RO number) and a sections array.
Each section has: section_id, prefix (ASI/BWO/CSI/JSI/WSI/ISI), page_count, header, footer, content.

Header fields: ro_number, vin, customer_name, customer_address, customer_city_state_zip,
home_phone, contact_phone, advisor_id, advisor_name, color, year, make, model, submodel,
engine, miles_in, miles_out, tag, open_date, close_date, deal_type, pay_type, rate,
stock_number, unit_number, customer_number, time_in, date_in, time_out, date_out.

Footer fields: labor_amount, parts_amount, gas_oil_lube, sublet_amount, misc_charges,
total_charges, less_insurance, sales_tax, please_pay.

Use null for missing fields. Numbers exact. Return ONLY the JSON object.

### doc
678856

## Generated Outputs
{
  "doc_id": "678

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Iteration 4: Proposed new text for system_prompt: You are extracting structured data from BMW repair order PDFs.

Return a JSON object with `doc_id` (the RO number) and a `sections` array.
Each section has:
- `section_id`: A string formatted as `prefix-section_id` where `prefix` is one of `ASI`, `BWO`, `CSI`, `JSI`, `WSI`, or `ISI`, and `section_id` is a unique identifier for that section.
- `prefix`: A string representing the section type, which can be one of `ASI`, `BWO`, `CSI`, `JSI`, `WSI`, or `ISI`.
- `page_count`: An integer indicating the number of pages in the section.
- `header`: A dictionary containing key-value pairs for the following header fields:
  - `ro_number`: A string representing the RO number (e.g., `629903`).
  - `vin`: A string representing the vehicle identification number (VIN).
  - `customer_name`: A string representing the name of the customer.
  - `customer_address`: A string representing the address of the customer.
  - `customer_city_state_zip`: A string re

Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Iteration 5: Proposed new text for system_prompt: You are extracting structured data from BMW repair order PDFs.

Return a JSON object with doc_id (the RO number) and a sections array.
Each section has: section_id, prefix (ASI/BWO/CSI/JSI/WSI/ISI), page_count, header, footer, content.

Header fields: ro_number, vin, customer_name, customer_address, customer_city_state_zip, home_phone, contact_phone, advisor_id, advisor_name, color, year, make, model, submodel, engine, miles_in, miles_out, tag, open_date, close_date, deal_type, pay_type, rate, stock_number, unit_number, customer_number, time_in, date_in, time_out, date_out.

Footer fields: labor_amount, parts_amount, gas_oil_lube, sublet_amount, misc_charges, total_charges, less_insurance, sales_tax, please_pay.

Use null for missing fields. Numbers exact. Return ONLY the JSON object.
``` ```
```
You are extracting structured data from BMW repair order PDFs.

Return a JSON object with doc_id (the RO number) and a sections array.
Each se

## 6. Results

In [11]:
optimized = run_eval(samples, best_prompt, label='optimized')

print('\n' + '-'*46)
print('  Doc          Baseline   Optimized     Delta')
print('-'*46)
for b, o in zip(baseline, optimized):
    d    = o['score'] - b['score']
    sign = '+' if d >= 0 else ''
    print('  ' + b['doc_id'].ljust(12) + str(round(b['score'],3)).rjust(8) +
          str(round(o['score'],3)).rjust(11) + (sign+str(round(d,3))).rjust(10))
avg_b = sum(r['score'] for r in baseline)  / len(baseline)
avg_o = sum(r['score'] for r in optimized) / len(optimized)
d     = avg_o - avg_b
print('-'*46)
print('  AVERAGE      ' + str(round(avg_b,3)).rjust(5) + str(round(avg_o,3)).rjust(14) +
      ('+' if d>=0 else '')+str(round(d,3)).rjust(9))
print('\n' + ('+' if d>=0 else '') + str(round(d*100,1)) + '% from prompt optimization. Same model, same docs.')

[optimized] 678856... FAILED
[optimized] 344098... FAILED
[optimized] 629903... score=0.862
[optimized] 809570... FAILED
[optimized] 201414... score=0.838
[optimized] 944962... FAILED

average: 0.2833

----------------------------------------------
  Doc          Baseline   Optimized     Delta
----------------------------------------------
  201414         0.838        0.0    -0.838
  344098           0.0        0.0      +0.0
  629903         0.862      0.862      +0.0
  678856           0.0        0.0      +0.0
  809570           0.0      0.838    +0.838
  944962           0.0        0.0      +0.0
----------------------------------------------
  AVERAGE      0.283         0.283+      0.0

+0.0% from prompt optimization. Same model, same docs.
